# Phase 4+: 归因分析 (Attribution Modeling) - 谁是真正的功臣？🦸‍♂️

> **🎯 目标**: 解决 Marketing / Operation 中最经典的争吵 —— "这 100 万业绩到底是哪个广告带来的？"
> **关键点**: **Markov Chain (马尔可夫链)** —— 在多渠道 (Multi-Touch) 时代，告别拍脑袋分钱。

## 1. 为什么需要归因？
现在的用户路径太复杂了：
1.  先在 **抖音** 刷到了广告 (Brand Awareness)。
2.  去 **百度** 搜了一下 (Search)。
3.  最后收到了 **短信** 优惠券，点击购买 (Conversion)。

**钱怎么分？**
*   **Last Touch**: 全给短信？(抖音委屈：我也出力了啊！)
*   **Linear**: 大家平分？(百度可能只是随便搜搜，不值这么多)。
*   **Data-Driven (Markov)**: 让数据说话，看谁真正提升了转化率。

---
### 0. 准备数据
我们来造一些 "用户旅程" (User Journey) 数据。

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# 1. 模拟用户路径 (Path Data)
# 格式: Channel1 > Channel2 > ... > Conversion/Null
paths = [
    'Douyin > Search > SMS > Conversion', # 完整路径
    'Douyin > Search > SMS > Conversion',
    'Douyin > Search > Null',             # 看了广告没买
    'Douyin > Search > Null',
    'Search > SMS > Conversion',          # 没看抖音直接搜
    'Douyin > SMS > Conversion',          # 没搜直接买
    'Douyin > Null',                      # 只看了抖音
    'SMS > Conversion'                    # 只看了短信
] * 100 # 放大一点数据量

df_paths = pd.DataFrame({'Path': paths})
print(df_paths.head())

## 2. 传统模型 (Heuristic Models)
先来看看以前大家是怎么拍脑袋的。

In [ ]:
def calculate_last_touch(df, channel_name):
    # 简单逻辑: 只要最后一步是这个 channel 且转化了，就算它的
    conversions = df[df['Path'].str.endswith(f'{channel_name} > Conversion')].shape[0]
    return conversions

print(f"SMS Last Touch Conversions: {calculate_last_touch(df_paths, 'SMS')}")
print(f"Douyin Last Touch Conversions: {calculate_last_touch(df_paths, 'Douyin')}")
# 结果肯定 SMS 赢麻了，因为它是收割位。Douyin 只有 0。

## 3. 马尔可夫链 (Markov Chain) - 真的黑科技 🕷️
原理非常简单：**Removal Effect (移除效应)**。

*   **逻辑**: 如果我把 **"Douyin"** 这个节点从全世界**删掉**，整体的转化率会跌多少？
    *   如果跌了很多 -> 说明 Douyin 很重要。
    *   如果没啥变化 -> 说明 Douyin 可有可无。

我们使用 `pomegranate` 库或者手写简单的转移矩阵来实现。
这里为了教学直观，我们用 pandas 手算 **转移概率矩阵 (Transition Matrix)**。

In [ ]:
# 1. 拆解所有 Step
# Start -> Douyin -> Search -> ...

transitions = []
for pid, path in enumerate(df_paths['Path']): # 加个 enumerate 拿 ID
    steps = ['Start'] + path.split(' > ')
    for i in range(len(steps)-1):
        transitions.append({
            'Journey_ID': pid,  # ✨ 加上这个 Session ID
            'from': steps[i],
            'to': steps[i+1]
        })

df_trans = pd.DataFrame(transitions)
print(df_trans.head(10)) # 看看前10行，现在清晰多了

# 2. 计算转移概率 (得票率)
# 比如: 在 Douyin 之后，有多少人去了 Search，有多少人直接 Null 了？
trans_matrix = pd.crosstab(df_trans['from'], df_trans['to'], normalize='index')

# 画个热力图看看流量是怎么流转的
plt.figure(figsize=(10, 6))
sns.heatmap(trans_matrix, annot=True, cmap='Blues', fmt='.2f')
plt.title("Transition Probability Matrix (Markov Graph)")
plt.show()

### 💡 解读热力图
这张图就是数据驱动的证据！
1.  看 `Douyin` 行: 它通向 `Search` 的概率是多少？(可能是 0.5)
2.  看 `Search` 行: 它通向 `Conversion` 的概率是多少？

如果 `Douyin -> Search` 的概率很高，而 `Search -> Conversion` 也很高。
那么我们可以说：**Douyin 是转化的重要助攻王**。
哪怕 Last Touch 没它的份，它的权重也应该很高！

## 练习 (Homework) 🛠️
试着对比一下:
1.  如果只看 Last Touch，SMS 拿了多少功劳？
2.  如果看这个转移矩阵，Douyin 虽然不直接连接 Conversion，但它给 Search 输送了多少弹药？

这就是 **Data Analyst** 在预算分配会议上最大的价值。

---

## 4. Bonus: 异动归因 (Metric Attribution / Contribution Analysis)
> **场景**: 老板问 "昨天 GMV 跌了 10%，到底是流量跌了，还是转化率跌了？"
> **区别**: 刚才讲的是 **Marketing Attribution** (分广告费)，现在讲的是 **Metric Decomposition** (分锅)。这正是您在 前司某电商平台 提到的 "Z-score 贡献度拆解" 的核心场景。

### 核心公式 (Additive Decomposition)
假设 $GMV = Traffic \times CVR$
我们想知道 $\Delta GMV$ 中，Traffic 贡献了多少？CVR 贡献了多少？

我们可以写一个简单的 Python 函数来计算 **Contribution Impact**。

In [9]:
def attribution_decomposition(base_metrics, current_metrics):
    """
    简单两因素归因: y = a * b
    Delta_y ~ (Delta_a * b_base) + (Delta_b * a_base) + (Interaction)
    """
    traffic_base, cvr_base = base_metrics['Traffic'], base_metrics['CVR']
    traffic_curr, cvr_curr = current_metrics['Traffic'], current_metrics['CVR']
    
    gmv_base = traffic_base * cvr_base
    gmv_curr = traffic_curr * cvr_curr
    delta_gmv = gmv_curr - gmv_base
    
    # 1. Traffic 带来的变动 (假设 CVR 不变)
    traffic_impact = (traffic_curr - traffic_base) * cvr_base
    
    # 2. CVR 带来的变动 (假设 Traffic 不变)
    cvr_impact = (cvr_curr - cvr_base) * traffic_base
    
    # 3. 交互项 (共同影响，通常按比例分配或单独列出)
    interaction = delta_gmv - (traffic_impact + cvr_impact)
    
    print(f"GMV Change: {delta_gmv:.2f}")
    print(f"  - Due to Traffic: {traffic_impact:.2f} ({traffic_impact/delta_gmv:.1%})")
    print(f"  - Due to CVR:     {cvr_impact:.2f} ({cvr_impact/delta_gmv:.1%})")
    print(f"  - Interaction:    {interaction:.2f} (Residual)")

# Case: 流量大跌，但转化率微涨，总体 GMV 还是跌了
base = {'Traffic': 1000, 'CVR': 0.10} # GMV = 100
curr = {'Traffic': 800,  'CVR': 0.11} # GMV = 88

attribution_decomposition(base, curr)

GMV Change: -12.00
  - Due to Traffic: -20.00 (166.7%)
  - Due to CVR:     10.00 (-83.3%)
  - Interaction:    -2.00 (Residual)
